# 04. Оценка сохранённых экспериментов
Загрузите папку эксперимента с весами, данными и параметрами нормализации. Ноутбук строит таблицы и графики по сохранённым моделям.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "tsprep").is_dir() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the tsprep project")
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)


In [ ]:
import json
import pandas as pd
from tsprep.comparison_experiment import validate_completed_experiment
# Для новых результатов укажите папку v3 и отключите режим чтения прежних запусков.
EXPERIMENT_DIR = ROOT / "results" / "study_20261103T184252995031Z"
ALLOW_HISTORICAL_REEVALUATION = True
runs = validate_completed_experiment(EXPERIMENT_DIR, allow_legacy=ALLOW_HISTORICAL_REEVALUATION)
print(EXPERIMENT_DIR, "historical reevaluation:", ALLOW_HISTORICAL_REEVALUATION)
runs.groupby(["dataset", "model"])[["mse", "forecast_mae", "forecast_vs_mean", "mse_vs_pca"]].agg(["mean", "std"])


In [ ]:
from tsprep.experiment import load_artifact, evaluate_artifact
# Пример задан заранее, без выбора по тестовой ошибке.
selected = runs[(runs.dataset == "aapl") & (runs.model == "tcn_ae")]
if selected.empty:
    selected = runs.sort_values(["dataset", "fold", "model", "seed"])
row = selected.iloc[0]
print("Reevaluation example:", row.dataset, row.fold, row.model, row.seed)
ARTIFACT = EXPERIMENT_DIR / row.artifact
metrics = evaluate_artifact(ARTIFACT, device="cpu", allow_source_mismatch=ALLOW_HISTORICAL_REEVALUATION)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
assert np.isclose(metrics["mse"], row.mse, rtol=1e-4, atol=1e-6)


In [ ]:
from tsprep.evaluation.reconstruction import collect_embeddings
from tsprep.data_preprocessing.datasets import build_eval_loader
from tsprep.evaluation.visualize import plot_loss_curves, plot_reconstruction_examples, plot_latent_projection
from tsprep.evaluation.latent_space import pca_project
from tsprep.mlcore.trainer import History
model, prepared, exp, manifest, arrays = load_artifact(ARTIFACT, allow_source_mismatch=ALLOW_HISTORICAL_REEVALUATION)
plot_loss_curves(History(**manifest["history"]), title=exp.model.model_name)
out = collect_embeddings(model, build_eval_loader(prepared.test_windows, exp.data.batch_size))
plot_reconstruction_examples(out["x"], out["x_hat"])
labels = None if prepared.regime is None else prepared.regime[prepared.window_starts["test"] + exp.data.window_len - 1]
plot_latent_projection(pca_project(out["z"]), labels, title="Descriptive PCA of embeddings")


## Как читать результаты
- Сравнивайте с PCA, Ridge по исходному окну, средним и медианой обучающей выборки.
- При одном классе sensitivity/ROC-AUC ограничены или не определены.
- Silhouette с истинными метками не заменяется кластеризацией KMeans.
- Отношение дисперсий зависит от масштаба z и не измеряет информацию.
- Порог аномалий выбирается на проверочной выборке; окна и их концы оцениваются отдельно.
- USREC задаёт ретроспективную метку периода.
- Ошибка к чистому эталону показывает удаление шума отдельно от восстановления зашумлённого входа.


In [ ]:
ALLOW_DOWNSTREAM_SOURCE_MISMATCH = ALLOW_HISTORICAL_REEVALUATION

## Прогноз, классификация и методы сравнения
Сводки разделены по набору данных и тестовому периоду. Среднее и стандартное отклонение показывают разброс между инициализациями, а не между независимыми экономическими историями. Каждый метод сравнения добавлен один раз для периода. `count` показывает число доступных измерений; пустая метрика не означает нулевое качество.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print
from tsprep.evaluation.experiment_results import load_experiment_tables, task_summary
from tsprep.evaluation.visualize import plot_task_metric

def show_task(frame, metrics):
    if frame.empty:
        print("Нет данных для этой задачи. Отсутствующая метрика не заменяется нулём.")
    else:
        display(task_summary(frame, metrics))

In [ ]:
experiment_tables = load_experiment_tables(EXPERIMENT_DIR, allow_source_mismatch=ALLOW_DOWNSTREAM_SOURCE_MISMATCH)
display(experiment_tables["availability"])
display(experiment_tables["support"])

### Прогнозирование
Ridge обучается на эмбеддингах обучающей выборки; коэффициент регуляризации выбирается на проверочной. Цель задаётся одним будущим преобразованным наблюдением в t+H, не ценой и не суммой доходностей. Для сравнения используются среднее, медиана, последнее наблюдение, Ridge по исходному окну и PCA+Ridge. Отрицательная `delta_mae_vs_*` означает преимущество перед соответствующим методом.


In [ ]:
show_task(experiment_tables["forecast"], ["mae", "r2", "validation_mae", "delta_mae_vs_mean",
                                    "delta_mae_vs_median", "delta_mae_vs_raw", "delta_mae_vs_pca"])
plot_task_metric(experiment_tables["forecast"], "mae", "Forecast MAE")
plt.show()

### Классификация и разделимость режимов
Бинарная классификация выполняется только при наличии нужных меток. Для трёх синтетических режимов считается silhouette. Классификация сравнивается с PCA и прогнозом самого частого обучающего класса. USREC задаёт ретроспективную метку периода, не публикацию в реальном времени. При одном классе в тесте AUC и часть других показателей не определены. Silhouette по истинным меткам и кластеризация KMeans решают разные диагностические задачи.


In [ ]:
show_task(experiment_tables["classification"], ["accuracy", "balanced_accuracy", "f1", "precision", "recall",
                                          "pr_auc", "roc_auc", "false_positive_rate", "n_positive", "n_negative"])
show_task(experiment_tables["clustering"], ["silhouette", "n_clusters"])

### Аномалии: окна и концы окон
Оценка всего окна = MSE + λ·KL, оценка конца окна = MSE последней точки. Порог выбирается только на проверочной выборке. Квантиль смешанной проверочной выборки не задаёт 5% FPR на нормальных точках. Для сравнения используются PCA и величина стандартизованного входа (raw magnitude). Метрики всего окна и последней точки считаются отдельно. Число эпизодов относится только к наблюдаемым концам окон с заданным шагом.


In [ ]:
show_task(experiment_tables["anomaly"], ["precision", "recall", "f1", "pr_auc", "false_positive_rate", "threshold",
    "positive_rate_test", "predicted_positive_rate", "n_positive", "n_negative", "n_events",
    "detected_events", "event_recall", "false_positive_points"])

### Восстановление, чистый эталон и латентное пространство
Чистый эталон доступен только на синтетике. Сравниваются MSE и MAE к эталону, исходный вход и PCA. Сохранённые ошибки к эталону усредняют все элементы перекрывающихся окон; разбор концов окон показан отдельно. Effective rank, active units и KL используются для диагностики. Отношение дисперсий зависит от масштаба и не измеряет информацию.


In [ ]:
show_task(experiment_tables["reconstruction"], ["mse", "mae", "r2"])
show_task(experiment_tables["clean"], ["mse", "mae", "r2", "delta_mse_vs_input", "delta_mae_vs_input"])
show_task(experiment_tables["latent"], ["effective_rank", "active_units", "kl", "variance_ratio"])

In [ ]:
from tsprep.evaluation.examples import downstream_example
from tsprep.mlcore.trainer import History
example_cache = {}

def example_for(dataset, fold=0, model="tcn_ae", seed=42):
    selected = runs[(runs.dataset == dataset) & (runs.fold == fold) & (runs.model == model) & (runs.seed == seed)]
    if selected.empty:
        print("Нет заданного примера:", dataset, fold, model, seed)
        return None
    path = EXPERIMENT_DIR/selected.iloc[0].artifact
    if str(path) not in example_cache:
        example_cache[str(path)] = downstream_example(path, device="cpu",
            allow_source_mismatch=ALLOW_HISTORICAL_REEVALUATION)
    return example_cache[str(path)]

## Прогнозы и ошибки относительно методов сравнения
Модель, период и инициализация примера заданы заранее, без выбора по тестовой ошибке. Показаны Ridge по z и Ridge по восстановленному окну. Для второго коэффициент регуляризации заново выбирается только на проверочной выборке; энкодер и декодер не дообучаются. Основные сохранённые показатели относятся к Ridge по z.


In [ ]:
forecast_example = example_for("aapl")
if forecast_example is None:
    forecast_example = example_for("synthetic")
if forecast_example is not None:
    f = forecast_example["forecast"]
    print(pd.DataFrame([{ "method": name, "mae": np.abs(f["target"]-pred).mean()}
                        for name, pred in f["predictions"].items()]).to_string(index=False))
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(f["time"], f["target"], label="target", linewidth=.8, alpha=.6)
    for name in ("embedding_ridge", "raw_window_ridge", "PCA_ridge", "train_median"):
        if name in f["predictions"]:
            ax.plot(f["time"], f["predictions"][name], label=name, linewidth=.9)
    ax.set_title("One transformed future observation at exact t+H")
    ax.legend()
    plt.show()
plot_task_metric(experiment_tables["forecast"], "delta_mae_vs_raw", "ΔMAE versus raw Ridge; negative = better")
plt.show()

## Классификация: матрица ошибок, PR-кривая и баланс классов
Порог LogisticRegression равен 0.5. PR-кривая и ROC-AUC имеют смысл только при наличии обоих классов в тесте. Постоянный прогноз нуля или самого частого класса может дать высокую accuracy, пропуская все положительные события. При отсутствии разметки классификация не выполняется; кластеры KMeans не считаются истинными метками.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve
class_example = example_for("aapl")
if class_example is not None and class_example["classification"] is not None:
    c = class_example["classification"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    ConfusionMatrixDisplay(confusion_matrix(c["labels"], c["prediction"], labels=[0, 1]), display_labels=[0, 1]).plot(ax=axes[0], colorbar=False)
    if len(np.unique(c["labels"])) == 2:
        precision, recall, _ = precision_recall_curve(c["labels"], c["scores"])
        axes[1].plot(recall, precision)
        axes[1].axhline(c["labels"].mean(), color="gray", linestyle="--", label="test prevalence")
        axes[1].set(xlabel="Recall", ylabel="Precision", title="Retrospective classification PR")
        axes[1].legend()
    else:
        axes[1].text(.1, .5, "Single test class: PR/AUC not identified")
    plt.show()
else:
    print("Бинарная классификация для выбранного примера недоступна.")
if not experiment_tables["classification"].empty:
    plot_task_metric(experiment_tables["classification"], "pr_auc", "Classification average precision")
    plt.show()
plot_task_metric(experiment_tables["clustering"], "silhouette", "Silhouette; see label_source")
plt.show()

## Аномалии с заранее выбранным порогом
Тестовые метки нужны только для оценки и обозначения событий на графике. Порог рассчитывается по проверочной выборке, без подбора по тестовой F1. Оценки всего окна и последней точки используют разную разметку. PR-AUC показывает качество ранжирования, а precision/recall при выбранном пороге показывают результат конкретного правила.


In [ ]:
anomaly_example = example_for("synthetic")
if anomaly_example is not None and anomaly_example["anomaly"] is not None:
    a = anomaly_example["anomaly"]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for row_index, unit in enumerate(("window", "endpoint")):
        scores, labels, threshold = a[unit+"_scores"], a[unit+"_labels"], a[unit+"_threshold"]
        time = anomaly_example["endpoint"]["time"]
        axes[row_index, 0].plot(time, scores, linewidth=.9)
        axes[row_index, 0].axhline(threshold, color="red", linestyle="--", label="validation threshold")
        axes[row_index, 0].scatter(time[labels == 1], scores[labels == 1], color="orange", s=15, label="true contamination")
        axes[row_index, 0].set_title(unit+" score; threshold frozen on validation")
        axes[row_index, 0].legend()
        if len(np.unique(labels)) == 2:
            precision, recall, _ = precision_recall_curve(labels, scores)
            axes[row_index, 1].plot(recall, precision)
            axes[row_index, 1].axhline(labels.mean(), color="gray", linestyle="--")
            axes[row_index, 1].set(xlabel="Recall", ylabel="Precision", title=unit+" PR")
        else:
            axes[row_index, 1].text(.1, .5, "Single test class")
    fig.tight_layout()
    plt.show()
else:
    print("Нет разметки аномалий для выбранного примера.")
plot_task_metric(experiment_tables["anomaly"], "f1", "Anomaly F1 at fixed validation threshold")
plt.show()
plot_task_metric(experiment_tables["anomaly"], "pr_auc", "Anomaly average precision")
plt.show()

## Ошибки на обычных точках чистого сигнала
Здесь учитывается один выход на каждый конец окна, без повторного подсчёта общих элементов. Это другое усреднение, чем у сохранённых ошибок всех элементов окон. Обычные и загрязнённые точки оцениваются отдельно; загрязнение включает импульс и его вклад в разность. Снижение MSE при росте MAE или повреждении обычных точек не считается общим улучшением удаления шума.


In [ ]:
if anomaly_example is not None and "clean" in anomaly_example["endpoint"]:
    e = anomaly_example["endpoint"]
    labels = anomaly_example["anomaly"]["endpoint_labels"].astype(bool)
    clean_rows = []
    for method in ("observed", "reconstructed", "PCA"):
        if method not in e:
            continue
        for subset, mask in (("all", np.ones(len(labels), bool)), ("normal", ~labels), ("contaminated", labels)):
            error = (e[method]-e["clean"])[mask]
            clean_rows.append(dict(method=method, subset=subset, n=int(mask.sum()),
                mse=float(np.square(error).mean()) if mask.any() else None,
                mae=float(np.abs(error).mean()) if mask.any() else None))
    display(pd.DataFrame(clean_rows))
    fig, ax = plt.subplots(figsize=(13, 4))
    for method in ("clean", "observed", "reconstructed", "PCA"):
        if method in e:
            ax.plot(e["time"][:160], e[method][:160, 0], label=method, alpha=.8)
    ax.set_title("Synthetic endpoints in the same standardized transformed scale")
    ax.legend()
    plt.show()
plot_task_metric(experiment_tables["clean"], "mae", "Clean MAE; observed_input is the no-processing baseline")
plt.show()

## Парные интервалы и ограничения
Для заданной TCN-AE считаются парные интервалы ошибок прогноза по всем доступным инициализациям, отдельно для каждого периода. Обучение не запускается. Если временных блоков недостаточно, границы не выдаются. Все модели можно разобрать через `scripts/analyze_experiment.py`, указав каталог эксперимента. Разброс по инициализациям и периодам не заменяет временной доверительный интервал. Интервалы относятся к выбранной истории; поправки на все исследовательские сравнения нет.


In [ ]:
from tsprep.evaluation.uncertainty import paired_block_interval
interval_rows = []
for (dataset, fold), group in runs[runs.model == "tcn_ae"].groupby(["dataset", "fold"]):
    paired_errors = {"train_median": [], "raw_window_ridge": [], "PCA_ridge": []}
    reference_target = None
    for seed in sorted(group.seed.unique()):
        example = example_for(dataset, fold=int(fold), seed=int(seed))
        f = example["forecast"]
        if reference_target is not None:
            np.testing.assert_array_equal(reference_target, f["target"])
        reference_target = f["target"]
        neural_error = np.abs(f["target"]-f["predictions"]["embedding_ridge"])
        for baseline in paired_errors:
            if baseline in f["predictions"]:
                paired_errors[baseline].append(neural_error-np.abs(f["target"]-f["predictions"][baseline]))
    for baseline, deltas in paired_errors.items():
        if len(deltas) != len(group.seed.unique()):
            continue
        for block in (24, 48, 96):
            interval_rows.append(dict(dataset=dataset, fold=fold, model="tcn_ae", baseline=baseline,
                **paired_block_interval(np.stack(deltas), block, repeats=1000)))
display(pd.DataFrame(interval_rows))

In [ ]:
ANALYSIS_DIR = None  # Example: ROOT / "docs" / "results_analysis_2026-11-04"
if ANALYSIS_DIR is None:
    print("Каталог анализа не выбран. Для временных интервалов запустите scripts/analyze_experiment.py отдельно.")
else:
    from tsprep.provenance import read_json, file_hash
    analysis_protocol = read_json(ANALYSIS_DIR/"analysis_protocol.json")
    if analysis_protocol["origin_table_sha256"] != file_hash(EXPERIMENT_DIR/"comparison_runs.csv"):
        raise ValueError("Analysis belongs to another experiment/table")
    intervals = pd.read_csv(ANALYSIS_DIR/"paired_block_intervals.csv")
    display(intervals)
    display(intervals.groupby(["dataset", "status"]).size().rename("comparisons"))